In [41]:
import os
from dotenv import load_dotenv

# langchain
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import JinaEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain.agents import create_agent

In [4]:
load_dotenv()

True

In [6]:
groq_key = os.getenv("GROQ_API_KEY")
jina_key = os.getenv("JINA_API_KEY")

LOADING OUR DATA

In [10]:
DATA_FILE_PATH = os.path.join("data", "hr_policy.txt")

DATA INGESTION

In [11]:
loader = TextLoader(DATA_FILE_PATH, encoding="utf-8")

documents = loader.load()

print("DATA LOADED")
print('=='*40)
print(documents)

DATA LOADED
[Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)\n\n1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.\n\n2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM to 4 PM.\n\n3. PROBATION PERIOD\nAll new employees undergo a probation period of 3 months from their date of joining.\nDu

LANGCHAIN DOCUMENT

Langchain processes everything in the form of documents

Documents:
Page Content -- actual data
Metadata - extra information abou the data


In [12]:
len(documents)

1

In [13]:
print(documents[0].page_content)

COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)

1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

2. WORK FROM HOME POLICY
Employees may work from home up to 2 days per week, subject to manager approval.
Fully remote work arrangements require written approval from the department head.
Employees working from home must be reachable during core hours: 10 AM to 4 PM.

3. PROBATION PERIOD
All new employees undergo a probation period of 3 months from their date of joining.
During probation, employees are not eligible for paid leave, but may take unpaid leave
in case of e

In [14]:
print(documents[0].metadata)

{'source': 'data\\hr_policy.txt'}


In [17]:
print(f'Total Characters: {len(documents[0].page_content)}')

Total Characters: 2597


SPLITTING OUR DATA

In [20]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print(chunks)

[Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)'), Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.'), Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM

In [21]:
len(chunks)

9

NOW EACH SPLIT CHUNK IS A DOCUMENT

In [22]:
print(chunks[5].page_content)

5. REIMBURSEMENT POLICY
Employees can claim reimbursement for approved business expenses such as travel,
client meals, and internet bills used for official work.
All reimbursement claims must be submitted with valid bills within 30 days of the expense.
Claims are processed within 10 working days after approval from the reporting manager.


EMBEDD OUR DATA

In [29]:
embeddings_model = JinaEmbeddings(model_name="jina-embeddings-v2-base-en")

STORE DATA IN VECTOR DB


In [30]:
vector_store = FAISS.from_documents(chunks, embeddings_model)

print("CHUNKS ARE STORED", vector_store.index.ntotal)



CHUNKS ARE STORED 9


In [33]:
test_query = "How many sick leaves employees get"

## SIMILARITY SEARCH

top_matches = vector_store.similarity_search(test_query, k=2)
print(f"Query: {test_query}")

for i, match in enumerate(top_matches, start=1):
    print(f"match {i}")
    print(match.page_content)




Query: How many sick leaves employees get
match 1
1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.
match 2
7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.


# TOOL

In [43]:
retriever = vector_store.as_retriever(search_kwargs={"k":3})  # returns top 3 relevant chunks

def search_hr_policy(question:str)->str:
    """
    Search the HR policy document for information about leave, work from home, probation, notice period, reimbursement code of conduct, holidays, or exit process
    """
    # the above is necessary or llm to decide which tool to use
    matching_chunks = retriever.invoke(question)
    return "\n\n".join(chunk.page_content for chunk in matching_chunks)

# DATA RETRIEVAL

### LLM

In [63]:
llm = ChatGroq(
    model = "openai/gpt-oss-120b",
    temperature=0.3      # creativity
)

llm.model_name


'openai/gpt-oss-120b'

In [73]:
test_response = llm.invoke("What are http headers? How to use it properly. Give brief answer")

In [74]:
print(test_response.content)

**HTTP headers** are name‑value pairs that travel in the start line of an HTTP request or response.  
They convey metadata about the request (what the client wants) or the response (what the server is sending) without being part of the actual payload.

| Direction | Typical purpose | Common examples |
|-----------|----------------|-----------------|
| **Request** | Tell the server what the client can handle, who it is, how to authenticate, etc. | `Accept`, `Accept‑Language`, `User‑Agent`, `Authorization`, `Cookie`, `If-None-Match`, `Cache-Control` |
| **Response** | Describe the payload, control caching, indicate status, set cookies, etc. | `Content-Type`, `Content-Length`, `ETag`, `Set-Cookie`, `Cache-Control`, `Location`, `Server` |

---

### How to use them properly

1. **Follow the spec** – RFC 7230‑7235 (or the newer HTTP/2 & HTTP/3 specs). Use the exact header names and allowed values; browsers and servers are strict about spelling and case‑insensitivity.

2. **Send only what’s n

### AI AGENT

LLM - Brain

Tool - Super power

Memory - No memory

In [64]:
hr_assistant = create_agent(
    model = llm,
    tools = [search_hr_policy],
    system_prompt = """
    You are a friendly HR assistant.
    Always use the search_hr_policy tool to look up facts before answering.
    If the answer isn't in the search results, say you don't know instead of guessing.
    """
)

In [66]:
def ask_hr_assistant(question: str) -> str:
    """Send a question to the RAG agent and print a nicely formatted answer."""
    print('=='*60 )
    print("QUESTION: ", question)
    print("--"*60)

    response = hr_assistant.invoke({"messages":[{"role": "user", "content": question}]})
    answer = response["messages"][-1].content

    print("ANSWER: ", answer)
    print("=" * 60)
    print()
    return answer


In [67]:
response = hr_assistant.invoke(
    {
        "messages":[
            {
                "role":"user",
                "content":"tell me about leave policies and how to apply for leave"
            }
        ]
    }
)

SYSTEM MESSAGE - HR ASST

HUMAN MESSAGE - TELL ME ABOUT POLICIES

AI MESSAGE - HEY THESE ARE THE POLICIES

In [68]:
response

{'messages': [HumanMessage(content='tell me about leave policies and how to apply for leave', additional_kwargs={}, response_metadata={}, id='13b62e0f-4799-4e56-b6f2-535132a297f0'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to answer about leave policies and how to apply. Must use search_hr_policy tool to look up facts before answering. So we need to query about leave policies. Probably ask "What are the leave policies and how to apply for leave?" Let\'s call search_hr_policy.', 'tool_calls': [{'id': 'fc_0407056e-8b73-431e-a189-c44a27e46d5e', 'function': {'arguments': '{"question":"leave policies and how to apply for leave"}', 'name': 'search_hr_policy'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 91, 'prompt_tokens': 200, 'total_tokens': 291, 'completion_time': 0.193889542, 'completion_tokens_details': {'reasoning_tokens': 56}, 'prompt_time': 0.008807425, 'prompt_tokens_details': None, 'queue_time': 0.31820359, 'total_

In [70]:
print(response["messages"][-1].content)

**Leave Policies – at a glance**

| Type of leave | Entitlement | Key rules |
|---------------|------------|-----------|
| **Annual (paid) leave** | 20 days per calendar year (full‑time) | • Requests must be entered in the HR portal **at least 5 working days before the start date**.<br>• Up to **5 days** can be carried forward to the next year; any excess expires at year‑end. |
| **Sick leave** | 10 paid days per year | • Separate from annual leave.<br>• If you are off for **more than 2 consecutive days**, attach a medical certificate when you submit the request. |
| **Public holidays** | 12 days (company‑wide) | • Listed in the annual holiday calendar released by HR.<br>• If you work on a public holiday you receive compensatory leave. |

**How to apply for leave**

1. **Log in to the HR portal** – use your employee credentials.  
2. **Navigate to “Leave Management”** (or the “Leave Request” tab).  
3. **Select the leave type** (Annual, Sick, etc.) and choose the start and end dates.  